# Notebook 1: OpenFace Models (v4)

**Account:** A | **GPU:** T4 x2 | **Time:** ~15-19 hrs

**Datasets to attach:** `smartlms-openface-features` + `smartlms-scripts` (or whatever you named them)

Trains: Temporal Transformer, BiLSTM-GRU, XGBoost 5-Fold CV, Stacking Ensemble, CORAL Ordinal

In [ ]:
# ── Cell 1: Install dependencies ──
# IMPORTANT: numpy<2 is required — Kaggle ships numpy 2.x but pandas needs numpy.rec (removed in 2.0)
# After running this cell, RESTART the runtime (Runtime → Restart session), then run from Cell 2
!pip install -q "numpy<2" xgboost optuna shap transformers

import numpy as np
print(f"numpy: {np.__version__}")
if np.__version__.startswith("2"):
    raise RuntimeError("⚠ numpy 2.x still loaded! Restart runtime (Runtime → Restart session), then re-run from Cell 2")

import torch, os, sys, shutil, glob, subprocess
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
if torch.cuda.device_count() > 1:
    print(f"Multi-GPU: {torch.cuda.device_count()} GPUs — DataParallel enabled")

In [ ]:
# ── Cell 2: Copy scripts + auto-detect paths ──
WORK = "/kaggle/working"
os.makedirs(f"{WORK}/app/ml", exist_ok=True)
open(f"{WORK}/app/__init__.py", "w").close()
open(f"{WORK}/app/ml/__init__.py", "w").close()

# Auto-find scripts (handles nested folders & any dataset name)
def find_script(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return hits[0] if hits else None

scripts = [
    "train_model_v2.py", "train_model_v3.py", "train_model_v4.py",
    "train_multimodal_v5.py", "extract_face_embeddings.py",
    "train_videomae.py", "extract_audio_features.py"
]
for s in scripts:
    found = find_script(s)
    if found:
        shutil.copy(found, f"{WORK}/app/ml/{s}")
        print(f"✓ {s} ← {found}")
    else:
        print(f"✗ NOT FOUND: {s}")

sys.path.insert(0, WORK)
os.chdir(WORK)

In [ ]:
# ── Cell 3: Auto-detect dataset paths & patch scripts ──
print("Attached datasets:")
for d in sorted(os.listdir("/kaggle/input")):
    print(f"  /kaggle/input/{d}/")

# Find OpenFace CSVs
csvs = glob.glob("/kaggle/input/**/openface_output/*.csv", recursive=True)
if not csvs:
    all_csvs = glob.glob("/kaggle/input/**/*.csv", recursive=True)
    csvs = [c for c in all_csvs if 'Labels' not in c and 'metadata' not in c.lower()]
OPENFACE_DIR = os.path.dirname(csvs[0]) if csvs else None
print(f"\nOpenFace CSVs: {len(csvs)}")
print(f"  Directory: {OPENFACE_DIR}")

# Find Labels — prefer the one under a DAiSEE/ folder (from daisee-videos dataset)
result = subprocess.run(["find", "/kaggle/input", "-name", "AllLabels.csv"], capture_output=True, text=True)
labels_hits = [l.strip() for l in result.stdout.strip().split("\n") if l.strip()]
print(f"\nAllLabels.csv found at:")
for h in labels_hits:
    print(f"  {h}")

# Pick the right one: prefer path with /DAiSEE/ so scripts' path construction works
daisee_labels = [l for l in labels_hits if '/DAiSEE/' in l]
other_labels = [l for l in labels_hits if '/DAiSEE/' not in l]

if daisee_labels:
    LABELS_CSV = daisee_labels[0]
    LABELS_DIR = os.path.dirname(LABELS_CSV)
    # DAISEE_DIR must be the parent of the DAiSEE/ folder
    # e.g. /kaggle/input/daisee-videos/DAiSEE/Labels → DAISEE_DIR = /kaggle/input/daisee-videos
    idx = LABELS_CSV.find('/DAiSEE/')
    DAISEE_DIR = LABELS_CSV[:idx] if idx >= 0 else os.path.dirname(LABELS_DIR)
    print(f"\n✓ Using DAiSEE dataset labels (correct structure)")
elif other_labels:
    # Fallback: labels are not under a DAiSEE/ folder (e.g. inside openface-features)
    LABELS_CSV = other_labels[0]
    LABELS_DIR = os.path.dirname(LABELS_CSV)
    DAISEE_DIR = os.path.dirname(LABELS_DIR)
    print(f"\n⚠ Labels not under DAiSEE/ — will patch scripts to use direct path")
else:
    LABELS_DIR = None
    DAISEE_DIR = None
    print("\n✗ AllLabels.csv NOT FOUND!")

print(f"Labels dir:  {LABELS_DIR}")
print(f"DAiSEE dir:  {DAISEE_DIR}")

# Verify the expected path works
expected_labels = os.path.join(DAISEE_DIR, "DAiSEE", "Labels", "AllLabels.csv") if DAISEE_DIR else ""
path_ok = os.path.exists(expected_labels)
print(f"Path check:  DAISEE_DIR/DAiSEE/Labels/AllLabels.csv → {'✓ EXISTS' if path_ok else '✗ MISSING'}")

# ── Patch train_model_v4.py ──
v4_path = f"{WORK}/app/ml/train_model_v4.py"
code = open(v4_path).read()
if OPENFACE_DIR:
    code = code.replace(
        'OPENFACE_DIR = "/kaggle/input/smartlms-openface/openface_output"',
        f'OPENFACE_DIR = "{OPENFACE_DIR}"'
    )
if DAISEE_DIR:
    code = code.replace(
        'DAISEE_DIR = "/kaggle/input/daisee-dataset"',
        f'DAISEE_DIR = "{DAISEE_DIR}"'
    )
# Safety net: if DAISEE_DIR/DAiSEE/Labels doesn't exist, patch labels_dir directly
if not path_ok and LABELS_DIR:
    code = code.replace(
        'labels_dir = os.path.join(DAISEE_DIR, "DAiSEE", "Labels")',
        f'labels_dir = "{LABELS_DIR}"  # Auto-patched: direct path'
    )
open(v4_path, 'w').write(code)
print("\n✓ Patched train_model_v4.py")

# ── Patch train_multimodal_v5.py ──
v5_path = f"{WORK}/app/ml/train_multimodal_v5.py"
if os.path.exists(v5_path):
    code = open(v5_path).read()
    if OPENFACE_DIR:
        code = code.replace(
            'OPENFACE_DIR = "/kaggle/input/smartlms-openface/openface_output"',
            f'OPENFACE_DIR = "{OPENFACE_DIR}"'
        )
    if DAISEE_DIR:
        code = code.replace(
            'DAISEE_DIR = "/kaggle/input/daisee-dataset"',
            f'DAISEE_DIR = "{DAISEE_DIR}"'
        )
    if not path_ok and LABELS_DIR:
        code = code.replace(
            'labels_dir = os.path.join(DAISEE_DIR, "DAiSEE", "Labels")',
            f'labels_dir = "{LABELS_DIR}"  # Auto-patched: direct path'
        )
    open(v5_path, 'w').write(code)
    print("✓ Patched train_multimodal_v5.py")

# ── Patch train_model_v2.py ──
v2_path = f"{WORK}/app/ml/train_model_v2.py"
if os.path.exists(v2_path) and DAISEE_DIR:
    code = open(v2_path).read()
    code = code.replace(r'C:\Users\revan\Downloads\DAiSEE', DAISEE_DIR)
    if not path_ok and LABELS_DIR:
        code = code.replace(
            'labels_dir = os.path.join(DAISEE_DIR, "DAiSEE", "Labels")',
            f'labels_dir = "{LABELS_DIR}"  # Auto-patched: direct path'
        )
    open(v2_path, 'w').write(code)
    print("✓ Patched train_model_v2.py")

print(f"\n✅ Expected: ~8231 CSVs, Labels dir with AllLabels.csv")

In [ ]:
# ── Cell 4: Verify imports work ──
from app.ml.train_model_v2 import load_labels, DIMENSION_NAMES
labels = load_labels(os.path.join(LABELS_DIR, "AllLabels.csv"))
print(f"Loaded {len(labels)} labels")
print(f"Dimensions: {DIMENSION_NAMES}")
print("✅ Imports working!")

In [ ]:
# ── Cell 5: Train Temporal Transformer (~4 hrs) ──
# All 4 dimensions (boredom, engagement, confusion, frustration)
# Each dimension takes ~50-60 min on T4
from app.ml.train_model_v4 import run_v4_experiment

run_v4_experiment(mode="transformer", seq_len=60, transformer_epochs=120)

# Free GPU memory before next step
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print("\n✅ Transformer training complete!")

In [ ]:
# ── Cell 6: Train BiLSTM-GRU (~3 hrs) ──
from app.ml.train_model_v4 import run_v4_experiment

run_v4_experiment(mode="bilstm_v4", seq_len=60, bilstm_epochs=100)

import torch, gc
gc.collect()
torch.cuda.empty_cache()
print("\n✅ BiLSTM-GRU training complete!")

In [ ]:
# ── Cell 7: Train XGBoost 5-Fold CV (~3 hrs) ──
from app.ml.train_model_v4 import run_v4_experiment

run_v4_experiment(mode="xgboost_cv", n_folds=5, n_trials=30)

import gc
gc.collect()
print("\n✅ XGBoost CV training complete!")

In [ ]:
# ── Cell 8: Train Stacking Ensemble (~1 hr) ──
# Stacks the 3 OpenFace models above
from app.ml.train_model_v4 import run_v4_experiment

run_v4_experiment(mode="stacking")

import gc
gc.collect()
print("\n✅ Stacking ensemble complete!")

In [ ]:
# ── Cell 9: Train CORAL Ordinal Regression (~4 hrs) ──
# Uses train_multimodal_v5 — set sys.argv to simulate CLI args
import sys
_orig_argv = sys.argv
sys.argv = ['train_multimodal_v5.py', '--mode', 'ordinal']

from app.ml.train_multimodal_v5 import main as v5_main
v5_main()

sys.argv = _orig_argv

import torch, gc
gc.collect()
torch.cuda.empty_cache()
print("\n✅ CORAL ordinal training complete!")

In [ ]:
# ── Cell 10: Review results ──
import json

print("=" * 80)
print("WEEK 1 RESULTS — OpenFace Models")
print("=" * 80)

v3_baseline = {"XGBoost v3": 0.570, "BiLSTM v3": 0.537, "Ensemble v3": 0.563}

for rf in sorted(glob.glob("/kaggle/working/experiment_results/*.json")):
    print(f"\n{os.path.basename(rf)}")
    with open(rf) as f:
        data = json.load(f)
    for key, res in data.get('results', {}).items():
        f1m = res.get('test_f1_macro') or res.get('cv_f1_macro_mean') or res.get('stacking_f1_macro', 0)
        print(f"  {key}: F1m = {f1m:.4f}")

print(f"\nv3 baselines: {v3_baseline}")

In [ ]:
# ── Cell 11: Save outputs for download ──
shutil.make_archive("/kaggle/working/week1_models", 'zip', "/kaggle/working/trained_models")
shutil.make_archive("/kaggle/working/week1_results", 'zip', "/kaggle/working/experiment_results")

for f in glob.glob("/kaggle/working/*.zip"):
    size_mb = os.path.getsize(f) / 1e6
    print(f"{os.path.basename(f)}: {size_mb:.1f} MB")

print("\n✅ Download these zip files from the Output tab!")
print("You'll need the proba_*.npy files from trained_models/ for Week 3 fusion.")